# XSA Research Notebook

Compares four XSA attention variants on a small GPT, all under identical conditions.

| # | Mode | `xsa_mode` | α | Notes |
|---|------|-----------|---|-------|
| 1 | Standard Attention | `sa` | 0 | baseline, no projection |
| 2 | Static XSA | `xsa` | 1.0 | full orthogonal projection (paper) |
| 3 | Adaptive XSA | `adaptive` | learned/head | model discovers per-head strength |
| 4 | Gated XSA | `gated` | σ(Linear(x)) | input-conditioned, most expressive |

Each mode trains for `N_STEPS` steps on random-token sequences, then reports:
- Final training loss
- Parameter count
- Tokens/sec throughput
- (Adaptive mode) learned α distribution per head
- (Gated mode) mean gate activation statistics

## 0 · Setup & Config

In [1]:
import sys, os, time, math
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

# Ensure workspace root is on path
ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from nanochat.gpt import GPT, GPTConfig
from nanochat.common import COMPUTE_DTYPE, COMPUTE_DTYPE_REASON

print(f'PyTorch: {torch.__version__}')
print(f'Compute dtype: {COMPUTE_DTYPE}  ({COMPUTE_DTYPE_REASON})')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

PyTorch: 2.6.0a0+ecf3bae40a.nv25.01
Compute dtype: torch.bfloat16  (auto-detected: CUDA SM 121 (bf16 supported))
Device: cuda


In [2]:
# ── Shared hyperparameters for all runs ─────────────────────────────────────
# Keep the model small so the notebook runs without a GPU cluster.
# Scale up depth/n_embd for more serious experiments.

BASE_CFG = dict(
    sequence_len = 256,
    vocab_size   = 512,
    n_layer      = 8,
    n_head       = 4,
    n_kv_head    = 4,
    n_embd       = 256,
    window_pattern = 'SSSL',
)

BATCH_SIZE = 4           # sequences per step
SEQ_LEN    = 128         # tokens per sequence
N_STEPS    = 200         # training steps per run (increase for real experiments)
LR         = 3e-4        # simple fixed LR for fair comparison

torch.manual_seed(42)

In [ ]:
# ── Minimal training loop (no distributed, no Flash Attention requirement) ──

def make_model(extra_cfg: dict) -> GPT:
    """Build and initialise a GPT from BASE_CFG + extra_cfg overrides."""
    cfg = GPTConfig(**{**BASE_CFG, **extra_cfg})
    # Build on CPU first (meta device is used internally, then we call init_weights)
    with torch.device('meta'):
        model = GPT(cfg)
    model = model.to_empty(device=DEVICE)
    model.init_weights()
    return model


def synthetic_batch(vocab_size: int, batch: int, seq: int):
    """Random token ids; targets are next-token (causal LM objective).
    .contiguous() is necessary: slices of a (B, seq+1) tensor have stride seq+1
    in dim-0 but size seq in dim-1, making them non-contiguous."""
    x = torch.randint(0, vocab_size, (batch, seq + 1), device=DEVICE)
    return x[:, :-1].contiguous(), x[:, 1:].contiguous()


def train_run(label: str, extra_cfg: dict, n_steps: int = N_STEPS):
    """
    Train a model for n_steps and return a results dict.
    Uses plain AdamW for simplicity (not Muon) so all modes are on equal footing.
    """
    print(f'\n{"="*60}')
    print(f'  Run: {label}')
    print(f'  Config overrides: {extra_cfg}')
    print(f'{"="*60}')

    model = make_model(extra_cfg)
    model.train()

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)

    losses, step_times = [], []
    vocab = BASE_CFG['vocab_size']

    t0 = time.perf_counter()
    for step in range(n_steps):
        t_step = time.perf_counter()
        x, y = synthetic_batch(vocab, BATCH_SIZE, SEQ_LEN)
        optimizer.zero_grad(set_to_none=True)
        loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        step_times.append(time.perf_counter() - t_step)
        losses.append(loss.item())
        if (step + 1) % 50 == 0:
            print(f'  step {step+1:>4}/{n_steps}  loss={loss.item():.4f}')

    elapsed = time.perf_counter() - t0
    tokens_per_sec = (n_steps * BATCH_SIZE * SEQ_LEN) / elapsed

    result = dict(
        label        = label,
        n_params     = n_params,
        final_loss   = losses[-1],
        mean_loss    = sum(losses[-20:]) / 20,   # last-20 mean
        tokens_per_s = tokens_per_sec,
        losses       = losses,
        model        = model,
        cfg          = {**BASE_CFG, **extra_cfg},
    )
    print(f'  Done in {elapsed:.1f}s  ({tokens_per_sec:.0f} tok/s)')
    return result

print('Helpers ready.')

Helpers ready.


## 1 · Standard Attention (SA) — Baseline

In [4]:
result_sa = train_run(
    label     = '1. SA (baseline)',
    extra_cfg = dict(xsa_mode='sa'),
)


  Run: 1. SA (baseline)
  Config overrides: {'xsa_mode': 'sa'}
  Parameters: 7,078,122


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

## 2 · Static XSA  (α = 1.0, full orthogonal projection)

In [ ]:
result_xsa = train_run(
    label     = '2. XSA (α=1.0, all layers)',
    extra_cfg = dict(xsa_mode='xsa', xsa_alpha=1.0, xsa_layer_mode='all'),
)

### 2b · XSA variants — partial α & deep-layers-only
Quick ablation sweep without a full training run.

In [ ]:
result_xsa_half = train_run(
    label     = '2b. XSA (α=0.5, all layers)',
    extra_cfg = dict(xsa_mode='xsa', xsa_alpha=0.5, xsa_layer_mode='all'),
)

result_xsa_deep = train_run(
    label     = '2c. XSA (α=1.0, deep layers only)',
    extra_cfg = dict(xsa_mode='xsa', xsa_alpha=1.0, xsa_layer_mode='deep'),
)

result_xsa_deep_nove = train_run(
    label     = '2d. XSA (α=1.0, deep, VE disabled)',
    extra_cfg = dict(xsa_mode='xsa', xsa_alpha=1.0, xsa_layer_mode='deep', xsa_disable_ve=True),
)

## 3 · Adaptive XSA  (learned α per attention head)
Each head learns its own projection strength independently. Init at α=1.0, clamped to [0, 2] during forward.

In [ ]:
result_adaptive = train_run(
    label     = '3. Adaptive XSA (α learned/head, init=1.0)',
    extra_cfg = dict(xsa_mode='adaptive', xsa_alpha=1.0, xsa_layer_mode='all'),
)

In [ ]:
# ── Inspect learned α values per layer & head ──────────────────────────────
model_adaptive = result_adaptive['model']
n_layer = BASE_CFG['n_layer']
n_head  = BASE_CFG['n_head']

alpha_table = []
for i, block in enumerate(model_adaptive.transformer.h):
    attn = block.attn
    if attn.xsa_mode == 'adaptive':
        alphas = attn.xsa_alphas.clamp(0.0, 2.0).detach().cpu().tolist()
        for h, a in enumerate(alphas):
            alpha_table.append({'layer': i, 'head': h, 'alpha': round(a, 4)})

df_alpha = pd.DataFrame(alpha_table)
pivot = df_alpha.pivot(index='layer', columns='head', values='alpha')
print('Learned α per (layer, head):')
print(pivot.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(pivot.values, vmin=0, vmax=2, cmap='RdYlGn_r', aspect='auto')
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title('Adaptive XSA: learned α per (layer, head)\n0=no projection, 1=full XSA, 2=over-projection')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 4 · Gated XSA  (input-conditioned α per position × head)
Most expressive variant: the gate `σ(Linear(x))` produces a scalar per token per head, letting the model learn *when* and *how much* to suppress the self-value component.

In [ ]:
result_gated = train_run(
    label     = '4. Gated XSA (σ(Wx), all layers)',
    extra_cfg = dict(xsa_mode='gated', xsa_layer_mode='all'),
)

In [ ]:
# ── Probe gate activation statistics ──────────────────────────────────────
# Pass a batch through the gated model and capture gate outputs to understand
# what the gate distribution looks like after training.

model_gated = result_gated['model']
model_gated.eval()

gate_stats = []  # (layer, mean_gate, std_gate)

hooks = []
captured = {}

def make_hook(layer_idx):
    def hook(module, inp, out):
        # out is the linear pre-sigmoid output; apply sigmoid here
        gate_val = torch.sigmoid(out).detach().cpu()
        captured[layer_idx] = gate_val
    return hook

for i, block in enumerate(model_gated.transformer.h):
    attn = block.attn
    if attn.xsa_mode == 'gated':
        hooks.append(attn.xsa_gate.register_forward_hook(make_hook(i)))

with torch.no_grad():
    x_probe, _ = synthetic_batch(BASE_CFG['vocab_size'], 8, SEQ_LEN)
    model_gated(x_probe)

for h in hooks:
    h.remove()

print(f'{'Layer':<8} {'Mean gate α':<16} {'Std gate α':<16} {'Min':<10} {'Max'}')
print('-' * 60)
for layer_idx, gate_vals in sorted(captured.items()):
    m  = gate_vals.mean().item()
    s  = gate_vals.std().item()
    mn = gate_vals.min().item()
    mx = gate_vals.max().item()
    print(f'{layer_idx:<8} {m:<16.4f} {s:<16.4f} {mn:<10.4f} {mx:.4f}')
    gate_stats.append({'layer': layer_idx, 'mean': m, 'std': s})

model_gated.train()

## 5 · Results Comparison

In [ ]:
# ── Loss curves: all runs on one plot ──────────────────────────────────────
all_results = [
    result_sa,
    result_xsa,
    result_xsa_half,
    result_xsa_deep,
    result_xsa_deep_nove,
    result_adaptive,
    result_gated,
]

fig, ax = plt.subplots(figsize=(10, 5))
for r in all_results:
    ax.plot(r['losses'], label=r['label'])
ax.set_xlabel('Step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('XSA Mode Comparison — Training Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
rows = []
for r in all_results:
    cfg = r['cfg']
    rows.append({
        'Run'         : r['label'],
        'xsa_mode'    : cfg.get('xsa_mode', 'sa'),
        'xsa_alpha'   : cfg.get('xsa_alpha', '-'),
        'xsa_layers'  : cfg.get('xsa_layer_mode', '-'),
        'xsa_disable_ve': cfg.get('xsa_disable_ve', False),
        'Params'      : f"{r['n_params']:,}",
        'Final loss'  : f"{r['final_loss']:.4f}",
        'Last-20 mean': f"{r['mean_loss']:.4f}",
        'tok/s'       : f"{r['tokens_per_s']:.0f}",
    })

df = pd.DataFrame(rows).set_index('Run')
print(df.to_string())

In [ ]:
# ── Bar chart: final loss per run ─────────────────────────────────────────
labels = [r['label'] for r in all_results]
means  = [r['mean_loss'] for r in all_results]

# Colour first bar differently as baseline
colors = ['#888'] + ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(labels)), means, color=colors[:len(labels)])
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Last-20 mean loss')
ax.set_title('XSA Mode Comparison — Final Loss (lower is better)')
ax.bar_label(bars, fmt='%.4f', fontsize=7, padding=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6 · α Sweep — XSA with varying projection strength
Trains static XSA for a range of α values to find the optimal partial-removal point without running a full sweep.

In [ ]:
ALPHA_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
sweep_results = []

for alpha in ALPHA_VALUES:
    mode = 'sa' if alpha == 0.0 else 'xsa'
    r = train_run(
        label     = f'α={alpha}',
        extra_cfg = dict(xsa_mode=mode, xsa_alpha=alpha, xsa_layer_mode='all'),
        n_steps   = 100,   # shorter run for the sweep
    )
    sweep_results.append((alpha, r['mean_loss']))

alphas, sweep_losses = zip(*sweep_results)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, sweep_losses, 'o-', color='#2196F3', linewidth=2, markersize=7)
ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='α=1 (full XSA)')
ax.set_xlabel('α (projection strength)')
ax.set_ylabel('Last-20 mean loss')
ax.set_title('XSA α Sweep (all layers)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_alpha, best_loss = min(sweep_results, key=lambda x: x[1])
print(f'Best α: {best_alpha}  (loss={best_loss:.4f})')

## 7 · Layer-Scope Sweep — which layers benefit most from XSA?

In [ ]:
n_layer = BASE_CFG['n_layer']   # 8
layer_scope_cfgs = [
    ('all layers',   dict(xsa_layer_mode='all')),
    ('deep only',    dict(xsa_layer_mode='deep')),
    ('last 2',       dict(xsa_layer_mode=f'{n_layer-2},{n_layer-1}')),
    ('first 2',      dict(xsa_layer_mode='0,1')),
    ('middle 4',     dict(xsa_layer_mode=','.join(str(i) for i in range(2, 6)))),
]

scope_results = []
for label, extra in layer_scope_cfgs:
    r = train_run(
        label     = f'XSA – {label}',
        extra_cfg = dict(xsa_mode='xsa', xsa_alpha=1.0, **extra),
        n_steps   = 100,
    )
    scope_results.append((label, r['mean_loss']))

scope_labels, scope_losses = zip(*scope_results)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(scope_labels, scope_losses, color='#4CAF50')
ax.axhline(result_sa['mean_loss'], color='red', linestyle='--', label='SA baseline')
ax.set_ylabel('Last-20 mean loss')
ax.set_title('XSA (α=1.0) — Effect of Layer Scope')
ax.bar_label(bars, fmt='%.4f', fontsize=8, padding=2)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

## 8 · Model Introspection — parameter & FLOPs breakdown

In [ ]:
# Inspect the adaptive model: parameter counts per XSA group
m = result_adaptive['model']
sp = m.num_scaling_params()
print('Parameter breakdown (adaptive model):')
for k, v in sp.items():
    print(f'  {k:<24} {v:>10,}')

print()
# Per-layer adaptive alpha params (small but explicit)
n_adaptive_params = sum(
    block.attn.xsa_alphas.numel()
    for block in m.transformer.h
    if m.config.xsa_mode == 'adaptive' and block.attn.xsa_mode == 'adaptive'
)
print(f'  XSA adaptive alpha params: {n_adaptive_params} '
      f'({BASE_CFG["n_head"]} heads × {BASE_CFG["n_layer"]} layers = {BASE_CFG["n_head"]*BASE_CFG["n_layer"]})')

print()
print('FLOPs comparison:')
for r in [result_sa, result_xsa, result_adaptive, result_gated]:
    mo = r['model']
    flops = mo.estimate_flops()
    print(f'  {r["label"]:<45} {flops/1e6:.1f}M FLOPs/token')

## 9 · Inference Sanity Check
Generate a short token sequence from each trained model to confirm they work at inference time.

In [ ]:
for r in [result_sa, result_xsa, result_adaptive, result_gated]:
    m = r['model']
    m.eval()
    prompt = [1, 2, 3, 4, 5]   # arbitrary token ids
    generated = list(m.generate(prompt, max_tokens=10, temperature=1.0, seed=0))
    print(f"{r['label']:<45} → {prompt} + {generated}")
    m.train()